# Do NHL teams achieve their expected goals?

Expected goals (xG) estimate how many goals a team should score based on the quality of its shots. This project compares each team's share of expected goals to its share of actual goals across 18 seasons, 2008 to 2025, to see how closely the two track each other and which teams beat or miss their xG.

Data: team season files from MoneyPuck.com (regular season). The xG model is MoneyPuck's.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

In [2]:
#pulling one team season file per year, MoneyPuck labels a season by its start year so 2025 is 2025-26
#the header makes the request look like a browser, without it the site can block the download
url = "https://moneypuck.com/moneypuck/playerData/seasonSummary/{}/regular/teams.csv"

frames = []
for year in range(2008, 2026):
    frame = pd.read_csv(url.format(year), storage_options={"User-Agent": "Mozilla/5.0"})
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
print(raw.shape)
print(raw["situation"].unique())

(2770, 109)
<StringArray>
['other', 'all', '5on5', '4on5', '5on4']
Length: 5, dtype: str


Each team appears several times per season, once for each game situation (5on5, power play, and so on). The `all` situation covers every minute of play, so only those rows are kept, which leaves one row per team per season.

In [3]:
#keeping the all situation and only the columns needed
#the file has two columns named team, pandas renames the second one team.1 so "team" is the first
df = raw[raw["situation"] == "all"][["team", "season", "xGoalsFor", "xGoalsAgainst", "goalsFor", "goalsAgainst"]].copy()
print(df.shape)
print(df["team"].value_counts().sort_index())

(554, 6)
team
ANA    18
ARI    16
ATL     3
BOS    18
BUF    18
CAR    18
CBJ    18
CGY    18
CHI    18
COL    18
DAL    18
DET    18
EDM    18
FLA    18
L.A    13
LAK     5
MIN    18
MTL    18
N.J    13
NJD     5
NSH    18
NYI    18
NYR    18
OTT    18
PHI    18
PIT    18
S.J    13
SEA     5
SJS     5
STL    18
T.B    13
TBL     5
TOR    18
UTA     2
VAN    18
VGK     9
WPG    15
WSH    18
Name: count, dtype: int64


Some franchises show up under more than one code. Older seasons use T.B, L.A, N.J and S.J for teams coded TBL, LAK, NJD and SJS in later seasons, and Atlanta (ATL) is the same franchise as Winnipeg (WPG). These were merged so each franchise counts as one team. Utah (UTA) is kept separate from Arizona (ARI).

In [4]:
#standardizing the franchise codes
code_fixes = {"T.B": "TBL", "L.A": "LAK", "N.J": "NJD", "S.J": "SJS", "ATL": "WPG"}
df["team"] = df["team"].replace(code_fixes)

print(df.shape)
print(df["team"].nunique())

(554, 6)
33


In [5]:
#share of expected goals and share of actual goals, same formulas as the Power BI custom columns
df["xg_share"] = df["xGoalsFor"] / (df["xGoalsFor"] + df["xGoalsAgainst"])
df["goal_share"] = df["goalsFor"] / (df["goalsFor"] + df["goalsAgainst"])

#gap is goal share minus xG share, positive means the team scored more than its xG predicted
df["gap"] = df["goal_share"] - df["xg_share"]
print(df[["xg_share", "goal_share", "gap"]].describe())

         xg_share  goal_share         gap
count  554.000000  554.000000  554.000000
mean     0.499947    0.500122    0.000175
std      0.032525    0.045319    0.029135
min      0.395952    0.348894   -0.076176
25%      0.478896    0.470736   -0.019709
50%      0.501830    0.503307    0.000796
75%      0.521962    0.531651    0.019140
max      0.599400    0.633684    0.094539


In [6]:
#average share and gap for each team across all of its seasons
teams = df.groupby("team")[["xg_share", "goal_share", "gap"]].mean()

#counting the seasons each team finished above and below its xG
df["above"] = df["gap"] > 0
df["below"] = df["gap"] < 0
teams["seasons"] = df.groupby("team")["season"].count()
teams["seasons_above"] = df.groupby("team")["above"].sum()
teams["seasons_below"] = df.groupby("team")["below"].sum()

teams = teams.sort_values("gap", ascending=False)
print(teams.head(5).round(3))
print(teams.tail(5).round(3))

      xg_share  goal_share    gap  seasons  seasons_above  seasons_below
team                                                                    
BOS      0.519       0.546  0.028       18             14              4
WSH      0.505       0.529  0.025       18             15              3
NYR      0.499       0.521  0.022       18             15              3
STL      0.499       0.518  0.019       18             15              3
WPG      0.491       0.506  0.015       18             11              7
      xg_share  goal_share    gap  seasons  seasons_above  seasons_below
team                                                                    
PHI      0.505       0.491 -0.014       18              3             15
OTT      0.496       0.481 -0.015       18              5             13
CAR      0.529       0.509 -0.020       18              5             13
NJD      0.505       0.483 -0.021       18              5             13
SJS      0.511       0.489 -0.022       18         

In [7]:
#exact season counts for the teams named on the dashboard
print(teams.loc[["BOS", "COL", "SJS", "CAR"], ["seasons", "seasons_above", "seasons_below"]])

      seasons  seasons_above  seasons_below
team                                       
BOS        18             14              4
COL        18             12              6
SJS        18              3             15
CAR        18              5             13


In [8]:
#regression of goal share on xG share across all team seasons
model = smf.ols("goal_share ~ xg_share", data=df).fit()
print(model.summary())
print("slope:", model.params["xg_share"])
print("R squared:", model.rsquared)

                            OLS Regression Results                            
Dep. Variable:             goal_share   R-squared:                       0.589
Model:                            OLS   Adj. R-squared:                  0.588
Method:                 Least Squares   F-statistic:                     791.7
Date:                Thu, 08 Oct 2026   Prob (F-statistic):          1.02e-108
Time:                        09:14:34   Log-Likelihood:                 1174.9
No. Observations:                 554   AIC:                            -2346.
Df Residuals:                     552   BIC:                            -2337.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0346      0.019     -1.816      0.0

The dashboard scatter plots one point per team, each the average of up to 18 seasons, which smooths out single season noise and makes the relationship look tighter. The regression above uses every team season, so its R squared is lower and is the better measure of how closely xG tracks goals in a single season.

# Conclusion

Across 554 team seasons, a 1 percentage point increase in xG share was associated with a 1.07 percentage point increase in goal share (95% CI 0.995 to 1.144, R squared 0.59). The slope is consistent with a one for one relationship, and xG share accounts for about 59% of the variation in single season goal share.